# 競馬EVシステム：Colab運用画面

このノートブックは、**GitHubのコードをColabで実行し、予想・購入・結果データをGoogle Driveへ保存する**ための運用画面です。

最初に `GITHUB_REPO` を自分のリポジトリへ変更し、上から順番に実行してください。


## 0. 設定

- `GITHUB_REPO`：`ユーザー名/リポジトリ名`
- publicリポジトリなら `PRIVATE_REPO=False`
- 個人データは `DRIVE_DATA_DIR` に保存されます


In [ ]:
GITHUB_REPO = "YOUR_GITHUB_NAME/keiba-hybrid-system"
GITHUB_BRANCH = "main"
PRIVATE_REPO = False
DRIVE_DATA_DIR = "/content/drive/MyDrive/keiba-ev-data"
PROJECT_DIR = "/content/keiba-hybrid-system"
CONFIG_PATH = f"{PROJECT_DIR}/config/default.json"


## 1. GitHubから最新版を取得してインストール

privateリポジトリの場合だけ、実行中にGitHub Personal Access Tokenを入力します。トークンは画面に表示されず、ノートブックへ保存されません。


In [ ]:
import os
import shutil
import subprocess
from getpass import getpass
from pathlib import Path

if GITHUB_REPO.startswith("YOUR_"):
    raise ValueError("GITHUB_REPOを自分のGitHubユーザー名/リポジトリ名へ変更してください")

project = Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)

if PRIVATE_REPO:
    token = getpass("GitHub token: ")
    clone_url = f"https://{token}@github.com/{GITHUB_REPO}.git"
else:
    clone_url = f"https://github.com/{GITHUB_REPO}.git"

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, clone_url, PROJECT_DIR],
    check=True,
)
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", PROJECT_DIR], check=True)
print("コード取得・インストール完了:", PROJECT_DIR)


## 2. Google Driveを接続してデータ保存先を準備


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from keiba_ev.storage import initialize_data_store

data_root = initialize_data_store(DRIVE_DATA_DIR)
print("データ保存先:", data_root)
print("作成済みファイル:", [p.name for p in sorted(data_root.glob("*.csv"))])


## 3. ライブラリと設定を読み込む


In [ ]:
import io
import json
import pandas as pd
from IPython.display import display

from keiba_ev.config import load_config
from keiba_ev.pipeline import analyze_race
from keiba_ev.odds_parser import parse_wide_text, parse_quinella_text, parse_trio_text
from keiba_ev.ev import attach_odds_and_ev
from keiba_ev.staking import allocate_budget
from keiba_ev.validation import validate_ticket_plan
from keiba_ev.storage import append_rows
from keiba_ev.reporting import roi_summary, grouped_roi

config = load_config(CONFIG_PATH)
print(json.dumps(config, ensure_ascii=False, indent=2))


## 4. レース情報を入力

`race_id` は重複しない値にしてください。例：`20260714_FUK_03`。


In [ ]:
race = {
    "race_id": "20260714_EXAMPLE_01",
    "date": "2026-07-14",
    "course": "福島",
    "race_no": 1,
    "surface": "芝",
    "distance": 1200,
    "going": "稍重",
    "weather": "曇",
    "class_name": "未勝利",
    "budget_ceiling": 1500,
    "algorithm_version": config["algorithm_version"],
}
race


## 5. 各馬の採点を貼り付け

ChatGPTからCSV形式で受け取り、三重引用符の中へ貼り付けます。

- `ability`：0〜40
- `suitability`：0〜20
- `pace`：0〜20
- `training`：0〜10
- `paddock`：0〜10
- `win_prob_input`：明示的な勝率がある場合だけ0〜1で入力。空欄なら採点から暫定推定
- `win_odds`：単勝オッズ


In [ ]:
HORSES_CSV = """horse_no,horse_name,ability,suitability,pace,training,paddock,win_prob_input,win_odds
1,サンプルA,34,16,15,8,7,,5.0
2,サンプルB,31,15,14,7,8,,10.0
3,サンプルC,28,14,15,7,7,,18.0
4,サンプルD,26,15,13,6,6,,25.0
"""

horses = pd.read_csv(io.StringIO(HORSES_CSV))
horses.insert(0, "race_id", race["race_id"])
display(horses)


## 6. 能力順位と確率を計算

`probability_source` が `score_softmax...` の場合、勝率は暫定モデルです。


In [ ]:
analysis = analyze_race(horses, config)
horse_analysis = analysis["horses"].sort_values(["ability_rank", "win_rank"])
display(
    horse_analysis[
        [
            "horse_no", "horse_name", "total_score", "ability_rank",
            "win_prob", "win_rank", "probability_source", "win_odds"
        ]
    ]
)


## 7. オッズを貼り付け

購入しない券種は空欄のままで構いません。ワイドは下限オッズを使います。


In [ ]:
WIDE_TEXT = """
1-2 5.2-6.1
1-3 8.4-9.2
2-3 10.0-11.5
"""

QUINELLA_TEXT = """
1-2 12.4
1-3 20.8
2-3 25.0
"""

TRIO_TEXT = """
1 1-2-3 18.5
2 1-2-4 30.0
3 1-3-4 45.0
"""

wide_odds = parse_wide_text(WIDE_TEXT, basis=config["betting"]["wide_odds_basis"])
quinella_odds = parse_quinella_text(QUINELLA_TEXT)
trio_odds = parse_trio_text(TRIO_TEXT)

print("ワイド解析件数:", len(wide_odds))
print("馬連解析件数:", len(quinella_odds))
print("三連複解析件数:", len(trio_odds))


## 8. 単勝・ワイド・馬連・三連複のEVを計算


In [ ]:
win_odds = horses[["horse_no", "win_odds"]].copy()
win_odds["selection"] = win_odds["horse_no"].astype(int).astype(str)
win_odds = win_odds.rename(columns={"win_odds": "odds"})[["selection", "odds"]]

ev_tables = {}
ev_tables["単勝"] = attach_odds_and_ev(analysis["win"], win_odds)
if not wide_odds.empty:
    ev_tables["ワイド"] = attach_odds_and_ev(analysis["wide"], wide_odds)
if not quinella_odds.empty:
    ev_tables["馬連"] = attach_odds_and_ev(analysis["quinella"], quinella_odds)
if not trio_odds.empty:
    ev_tables["三連複"] = attach_odds_and_ev(analysis["trio"], trio_odds)

for bet_type, table in ev_tables.items():
    print(f"\n【{bet_type} EV上位】")
    display(table.head(15))


## 9. 券種横断で資金配分

予算は上限です。正のEV候補が不足する場合、全額を使い切りません。


In [ ]:
candidate_tables = []
for bet_type, table in ev_tables.items():
    temp = table.copy()
    temp.insert(0, "bet_type", bet_type)
    candidate_tables.append(temp)

all_candidates = pd.concat(candidate_tables, ignore_index=True)
betting = config["betting"]

plan = allocate_budget(
    all_candidates,
    budget=int(race["budget_ceiling"]),
    unit=int(betting["unit"]),
    ev_threshold=float(betting["ev_threshold"]),
    kelly_fraction=float(betting["kelly_fraction"]),
    max_tickets=int(betting["max_tickets"]),
    max_ticket_share=float(betting["max_ticket_share"]),
)

display(plan)
validation = validate_ticket_plan(
    plan,
    budget_ceiling=int(race["budget_ceiling"]),
    unit=int(betting["unit"]),
)
print(validation)

if plan.empty:
    print("購入条件を満たす馬券がありません。見送り候補です。")


## 10. レース前データをGoogle Driveへ保存

実際に購入する内容へ修正してから実行してください。二重登録を避けるため、同じセルを繰り返し実行しないでください。


In [ ]:
from pathlib import Path

data_root = Path(DRIVE_DATA_DIR)
append_rows(data_root / "races.csv", pd.DataFrame([race]))
append_rows(data_root / "horses.csv", horses)

if not plan.empty:
    bets_to_save = plan.rename(
        columns={
            "odds": "odds_at_purchase",
            "ev": "ev_at_purchase",
        }
    ).copy()
    bets_to_save.insert(0, "race_id", race["race_id"])
    bets_to_save["payout"] = 0
    bets_to_save["algorithm_version"] = race["algorithm_version"]
    save_columns = [
        "race_id", "bet_type", "selection", "stake", "odds_at_purchase",
        "model_prob", "ev_at_purchase", "payout", "algorithm_version"
    ]
    append_rows(data_root / "bets.csv", bets_to_save[save_columns])

print("レース前データを保存しました")


## 11. レース後の結果を保存

着順とメモを入力して実行します。払戻は `bets.csv` をDrive上で編集するか、次のセルで該当行を更新してください。


In [ ]:
result = {
    "race_id": race["race_id"],
    "first": 1,
    "second": 2,
    "third": 3,
    "notes": "サンプル結果。実際の内容へ変更してください。",
}
append_rows(Path(DRIVE_DATA_DIR) / "results.csv", pd.DataFrame([result]))
print("結果を保存しました")


## 12. 回収率を集計

`bets.csv` の `payout` を実際の払戻へ更新した後に実行してください。


In [ ]:
bets_history = pd.read_csv(Path(DRIVE_DATA_DIR) / "bets.csv")
if bets_history.empty:
    print("購入履歴がありません")
else:
    print(roi_summary(bets_history))
    display(grouped_roi(bets_history, ["bet_type"]))


## 13. 動作テスト

コード更新後やエラー発生時に実行します。


In [ ]:
import subprocess
subprocess.run(["python", "-m", "pytest", "-q"], cwd=PROJECT_DIR, check=True)
